# RAG Application with LangChain

## Introduction to RAG (Retrieval Augmented Generation)

**Retrieval Augmented Generation (RAG)** is a technique used to enhance the capabilities of Large Language Models (LLMs) by providing them with access to external data.

LLMs like GPT-4 are trained on vast amounts of public data, but they don't know about your *private* data (e.g., your company's internal documents, emails, or databases). RAG bridges this gap by:
1.  **Retrieving** relevant information from your data source based on the user's query.
2.  **Augmenting** the prompt sent to the LLM with this retrieved information.
3.  **Generating** a response that uses the retrieved context to answer the question accurately.

In this notebook, we will build a RAG application that can chat with PDF documents. We will use **LangChain**, a powerful framework for building LLM applications.

### Key Concepts:
*   **Document Loading**: Reading files (PDFs, text, etc.).
*   **Splitting**: Breaking documents into smaller, manageable chunks.
*   **Embeddings**: Converting text into numerical vectors that capture semantic meaning.
*   **Vector Store**: A database optimized for storing and searching these vectors.
*   **Retrieval**: Finding the most relevant chunks for a query.
*   **Generation**: Using an LLM to synthesize an answer.


In [ ]:
!pip install -U -q langchain langchain-community langchain-openai langchain-chroma pypdf gradio python-dotenv

## Environment Setup

We need to set up our API keys. This notebook supports both Google Colab and local execution.
*   **Colab**: Use `google.colab.userdata` to securely access keys stored in Colab Secrets.
*   **Local**: Use `python-dotenv` to load keys from a `.env` file.


In [ ]:
import os
import sys

# Set the model name
MODEL = "gpt-4o-mini"
db_name = "vector_db"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    print("Loaded API key from Google Colab Userdata")
except ImportError:
    # If running locally, load from .env file
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded API key from .env file (if present)")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not found in environment variables.")


## Imports

We will import the necessary modules from `langchain` and other libraries.


In [ ]:
import glob
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
import gradio as gr


## 1. Load Documents

We need to load our PDF documents. We will use `DirectoryLoader` to load all PDFs from a directory and `PyPDFLoader` to parse each file.


In [ ]:
# Define the path to your documents
# In Colab, you might upload files to a folder named 'pdfs' or similar
# For this example, we'll look in the current directory or a subdirectory
folders = glob.glob("pdfs/*.pdf") + glob.glob("*.pdf") # Adjust path as needed

# If no files found, let's create a dummy file for demonstration if needed, 
# but assuming user has files or will upload them.
if not folders:
    print("No PDF files found. Please upload some PDF files.")
else:
    print(f"Found {len(folders)} PDF files.")

documents = []
for file_path in folders:
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    for doc in docs:
        # Add metadata to track which file this came from
        doc.metadata["source_file"] = os.path.basename(file_path)
        documents.append(doc)

print(f"Loaded {len(documents)} pages from {len(folders)} files.")


## 2. Split Documents

LLMs have a context window limit (a maximum number of tokens they can process). Also, we want to retrieve only the specific parts of a document that are relevant to a query, not the whole book.

We use `RecursiveCharacterTextSplitter` to split the text into chunks.
*   **chunk_size**: The maximum size of each chunk (in characters).
*   **chunk_overlap**: The number of characters to overlap between chunks. This ensures that context isn't lost if a sentence is cut in the middle.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

chunks = text_splitter.split_documents(documents)
print(f"Split {len(documents)} pages into {len(chunks)} chunks.")

if chunks:
    print("\nExample Chunk Content:")
    print(chunks[0].page_content[:200] + "...")
    print("\nMetadata:", chunks[0].metadata)


## 3. Embeddings and Vector Store

**Embeddings** are numerical representations of text. Text with similar meanings will have similar embedding vectors.

We will use `OpenAIEmbeddings` to generate these vectors and store them in **Chroma**, a vector database. This allows us to perform "semantic search" — finding text that means the same thing as the query, even if it uses different words.


In [ ]:
# Initialize Embeddings
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Initialize Vector Store (Chroma)
# If the database already exists, we can delete it to start fresh or load it.
# For this notebook, we'll recreate it to ensure it matches our current docs.

if os.path.exists(db_name):
    import shutil
    shutil.rmtree(db_name) # Clean up old DB

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name
)

print(f"Vectorstore created with {vectorstore._collection.count()} documents.")


## 4. Create the Retrieval Chain

We will now create the RAG chain using **LangChain Expression Language (LCEL)**.

This involves two main parts:
1.  **History-Aware Retriever**: This component takes the user's latest question and the chat history, and reformulates the question to be standalone. For example, if I ask "What is it?" after talking about "SecLM", it rewrites the question to "What is SecLM?".
2.  **Question-Answer Chain**: This component takes the retrieved documents and the standalone question, and generates the final answer.


In [ ]:
# 1. Initialize LLM
llm = ChatOpenAI(temperature=0, model_name=MODEL)

# 2. Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 3. Contextualize Question Prompt
# This prompt helps the LLM understand the user's question in the context of previous messages.
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# 4. Answer Question Prompt
# This prompt instructs the LLM to answer the question using the retrieved context.
qa_system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, just say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# 5. Final RAG Chain
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)


## 5. Test the Chain

Let's test the chain with a simple query.


In [ ]:
chat_history = []

query = "What is the main topic of these documents?"
response = rag_chain.invoke({"input": query, "chat_history": chat_history})

print("Question:", query)
print("Answer:", response["answer"])

# Update history
chat_history.extend([HumanMessage(content=query), AIMessage(content=response["answer"])])


## 6. Interactive Chat with Gradio

We will use **Gradio** to create a user-friendly chat interface. This allows us to have a continuous conversation with our documents.


In [ ]:
def predict(message, history):
    # Convert Gradio history to LangChain format
    history_langchain_format = []
    for human, ai in history:
        history_langchain_format.append(HumanMessage(content=human))
        history_langchain_format.append(AIMessage(content=ai))
    
    response = rag_chain.invoke({"input": message, "chat_history": history_langchain_format})
    return response["answer"]

# Create Gradio Interface
gr.ChatInterface(
    predict,
    title="RAG Chat with PDF",
    description="Ask questions about your PDF documents.",
    examples=["Summarize the documents", "What are the key findings?"],
).launch(share=True, debug=True)


# Part 2: Chat with SQL

In this section, we will explore how to use LangChain to interact with a SQL database using natural language.


In [ ]:
import requests
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_sql_agent
from langchain.agents.agent_types import AgentType

# Download Sample Database (Chinook)
url = "https://github.com/laxmimerit/All-CSV-ML-Data-Files-Download/raw/refs/heads/master/db_samples/Chinook.db"
if not os.path.exists("Chinook.db"):
    response = requests.get(url)
    if response.status_code == 200:
        with open("Chinook.db", "wb") as file:
            file.write(response.content)
        print("Chinook.db downloaded.")
    else:
        print("Failed to download database.")

# Connect to Database
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
print("Tables:", db.get_usable_table_names())

# Create SQL Agent
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True
)

# Test the Agent
response = agent_executor.invoke("How many employees are there?")
print(response["output"])
